# Net-Payout-Based Duration

## Step 0: Setup, Imports, Pfade, Session


In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from pathlib import Path

# Speicherstruktur fuer Intermediate und Final Output
BASE_DIR = Path(
    "/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data"
)
BASE_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = BASE_DIR / "intermediate"
DATA_DIR.mkdir(exist_ok=True)

def save_parquet(df: pd.DataFrame, name: str):
    path = DATA_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved: {path}")

def load_parquet(name: str) -> pd.DataFrame:
    path = DATA_DIR / f"{name}.parquet"
    return pd.read_parquet(path)


## Step 1: Universum laden

In [2]:
# Load quarterly EURO500 net-payout panel
# This file is the single source for duration inputs.
euro500_netpayout = load_parquet("euro500_netpayout").copy()

if "firm_id" not in euro500_netpayout.columns:
    raise KeyError("euro500_netpayout must contain firm_id")
if "effective_date" not in euro500_netpayout.columns:
    raise KeyError("euro500_netpayout must contain effective_date")

# Ensure Module D field is present already at Step 1 input stage.
if "CashSTInvst" not in euro500_netpayout.columns:
    euro500_netpayout["CashSTInvst"] = np.nan

# Year is defined strictly from effective_date
euro500_netpayout["effective_date"] = pd.to_datetime(euro500_netpayout["effective_date"], errors="coerce")
euro500_netpayout["year"] = euro500_netpayout["effective_date"].dt.year

print(
    f"Loaded EURO500 net-payout panel: rows={len(euro500_netpayout):,}, "
    f"firm_id={euro500_netpayout['firm_id'].nunique():,}"
)
print("Step 1 check: CashSTInvst in input =", "CashSTInvst" in euro500_netpayout.columns)



Loaded EURO500 net-payout panel: rows=54,000, firm_id=1,166
Step 1 check: CashSTInvst in input = True


## Step 2:  Firm-year Masterpanel (ME, BE, A, Sales, NI, GP, Debt, Div, Buyback, CashSTInvst)


In [3]:
def build_masterpanel_firmyear(euro500_netpayout):
    """
    Build one row per firm_id-year for valuation/state construction.
    Keeps firm metadata (name) and accounting/value fields.
    """
    df = euro500_netpayout.copy()

    df["firm_id"] = df["firm_id"].astype(str).str.strip()
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df = df.dropna(subset=["firm_id", "year"]).copy()
    df["year"] = df["year"].astype(int)

    if "effective_date" in df.columns:
        df["effective_date"] = pd.to_datetime(df["effective_date"], errors="coerce")

    rename_map = {
        "mcap_eur": "ME",
        "Sales": "sales",
        "NetIncome": "net_income",
        "GrossProfit": "gross_profit",
        "Dividends": "dividends",
        "Buybacks": "buybacks",
    }
    existing_renames = {k: v for k, v in rename_map.items() if k in df.columns}
    if existing_renames:
        df = df.rename(columns=existing_renames)

    cols = [
        "firm_id", "name", "year", "effective_date",
        "ME", "BE", "assets", "sales", "net_income", "gross_profit", "debt",
        "dividends", "buybacks", "CashSTInvst",
    ]
    cols = [c for c in cols if c in df.columns]
    df = df[cols].copy()

    value_cols = [
        c for c in [
            "ME", "BE", "assets", "sales", "net_income", "gross_profit", "debt", "dividends", "buybacks", "CashSTInvst"
        ] if c in df.columns
    ]
    for c in value_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # one row per firm-year: take the earliest effective_date within the year
    # (Q1 preferred; if unavailable then Q2/Q3/Q4)
    # Use row-wise deduplication to preserve a real row (groupby.first is column-wise).
    sort_cols = ["firm_id", "year"] + (["effective_date"] if "effective_date" in df.columns else [])
    df = df.sort_values(sort_cols)
    df = df.drop_duplicates(subset=["firm_id", "year"], keep="first").reset_index(drop=True)

    df["dividends"] = df["dividends"].fillna(0.0) if "dividends" in df.columns else 0.0
    df["buybacks"] = df["buybacks"].fillna(0.0) if "buybacks" in df.columns else 0.0
    df["PO"] = df["dividends"] + df["buybacks"]

    df = df.sort_values(["firm_id", "year"]).reset_index(drop=True)
    lag_vars = [
        c for c in ["ME", "BE", "assets", "sales", "net_income", "gross_profit", "debt", "dividends", "buybacks", "CashSTInvst", "PO"]
        if c in df.columns
    ]
    for c in lag_vars:
        df[f"{c}_lag1"] = df.groupby("firm_id")[c].shift(1)

    df["dBE"] = df["BE"] - df["BE_lag1"]
    df["avgBE"] = 0.5 * (df["BE"] + df["BE_lag1"])
    df["avgAssets"] = 0.5 * (df["assets"] + df["assets_lag1"])

    df.loc[df["ME"] <= 0, "ME"] = pd.NA
    df.loc[df["BE"] <= 0, "BE"] = pd.NA
    df.loc[df["assets"] <= 0, "assets"] = pd.NA
    df.loc[df["sales"] <= 0, "sales"] = pd.NA
    df.loc[df["debt"] < 0, "debt"] = pd.NA

    return df




In [4]:
master = build_masterpanel_firmyear(euro500_netpayout)

master.head()

,firm_id,name,year,effective_date,ME,BE,assets,sales,net_income,gross_profit,...,net_income_lag1,gross_profit_lag1,debt_lag1,dividends_lag1,buybacks_lag1,CashSTInvst_lag1,PO_lag1,dBE,avgBE,avgAssets
0,FIRM0000001,Ahlers AG,1999,1999-01-01,142904483.073431,9.267012e+07,2.019414e+08,2.960978e+08,1.538477e+07,1.450279e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FIRM0000001,Ahlers AG,2000,2000-01-03,111992999.999999,1.042371e+08,2.644079e+08,3.053139e+08,1.530706e+07,1.489179e+08,...,1.538477e+07,1.450279e+08,5.410082e+07,1.002132e+07,0.0,2.011780e+07,1.002132e+07,1.156696e+07,9.845360e+07,2.331747e+08
2,FIRM0000001,Ahlers AG,2001,2001-01-01,204792999.999999,1.087722e+08,2.651038e+08,3.795033e+08,1.489240e+07,1.845043e+08,...,1.530706e+07,1.489179e+08,1.013191e+08,0.000000e+00,0.0,3.948196e+07,0.000000e+00,4.535159e+06,1.065047e+08,2.647559e+08
3,FIRM0000001,Ahlers AG,2002,2002-01-01,162137749.999999,1.084580e+08,2.591870e+08,3.797310e+08,1.285700e+07,1.837410e+08,...,1.489240e+07,1.845043e+08,9.915228e+07,1.104697e+07,0.0,2.113936e+07,1.104697e+07,-3.142348e+05,1.086151e+08,2.621454e+08
4,FIRM0000001,Ahlers AG,2003,2003-01-01,145914500.0,1.072090e+08,2.512660e+08,3.504880e+08,1.212600e+07,1.733020e+08,...,1.285700e+07,1.837410e+08,9.405300e+07,1.188800e+07,0.0,2.816600e+07,1.188800e+07,-1.249000e+06,1.078335e+08,2.552265e+08


In [5]:
checks = pd.DataFrame({
    
    # valuation
    "ME<=0": (master["ME"] <= 0),
    "BE<=0": (master["BE"] <= 0),
    "sales<=0": (master["sales"] <= 0),
    
    # growth
    "BE_lag1<=0": (master["BE_lag1"] <= 0),
    "assets_lag1<=0": (master["assets_lag1"] <= 0),
    "sales_lag1<=0": (master["sales_lag1"] <= 0),

    # profitability denominators
    "avgBE<=0": (master["avgBE"] <= 0),
    "avgAssets<=0": (master["avgAssets"] <= 0),

    # log1p conditions
    "1+PO/ME<=0": (1 + master["PO"]/master["ME"] <= 0),
    "1+(PO+dBE)/BE_lag1<=0": (1 + (master["PO"] + master["dBE"]) / master["BE_lag1"] <= 0),
    "1+NI/avgBE<=0": (1 + master["net_income"] / master["avgBE"] <= 0),
    "1+GP/avgAssets<=0": (1 + master["gross_profit"] / master["avgAssets"] <= 0),

    # leverage denominator
    "debt+ME<=0": (master["debt"] + master["ME"] <= 0),
})

checks.mean().sort_values(ascending=False)

BE_lag1<=0               0.004947
avgBE<=0                 0.004191
1+(PO+dBE)/BE_lag1<=0    0.003642
1+NI/avgBE<=0            0.003298
sales_lag1<=0            0.002886
1+PO/ME<=0               0.000756
1+GP/avgAssets<=0        0.000137
ME<=0                         0.0
BE<=0                         0.0
sales<=0                      0.0
assets_lag1<=0                0.0
avgAssets<=0                  0.0
debt+ME<=0                    0.0
dtype: Float64

### Data Validity Checks

Before constructing the state variables, I perform a set of diagnostic checks to verify that all transformations required for the logarithmic variables are well-defined. In particular, I examine whether the denominators used in the ratios are positive and whether expressions entering a $\log(1+x)$ transformation satisfy the condition $1+x>0$.

The results indicate that potential violations are very rare. The largest shares occur for lagged book equity and average book equity, each affecting less than 0.5% of the observations. Similarly, cases where the profitability-based transformations would be undefined (e.g. $1 + NI/avgBE \leq 0$ or $1 + (PO+\Delta BE)/BE_{t-1} \leq 0$) remain below 0.4% of the sample.

All other conditions are essentially always satisfied. In particular, market equity, book equity, sales, and assets are strictly positive for the entire sample.

These findings suggest that the accounting data are well-behaved and that only a very small fraction of observations will be excluded when constructing the state variables. Observations for which the required transformations are not defined are handled naturally by the `safe_log` and `safe_log1p` functions, which return missing values (`NaN`).

## Step 3: Construction of Firm-Level State Variables


In this step, we construct the **firm-level state variables** that summarize valuation, growth, profitability, and capital structure. These variables form the **state vector** used later in the VAR and valuation identity, following the methodology of the paper as closely as possible.

All state variables are:
- constructed at **annual (firm-year) frequency**
- based on **reported, consolidated accounting data**
- computed using **information available at time $t$** (with lags where required)
- transformed using logs or $\log(1+x)$ where appropriate, exactly as in the paper

Observations for which the underlying transformation is not well-defined (e.g. non-positive denominators) are set to missing (`NaN`) rather than forced or winsorized.

***

### 3.1 Valuation States

These variables capture how the firm is priced relative to fundamentals.

- **Book-to-Market**

$$
bm_{i,t} = \log\left(\frac{BE_{i,t}}{ME_{i,t}}\right)
$$

- **Payout Yield**

$$
py_{i,t} = \log\left(1 + \frac{PO_{i,t}}{ME_{i,t}}\right)
$$

Negative values correspond to net equity issuance rather than cash distributions.

- **Sales Yield**

$$
sy_{i,t} = \log\left(\frac{Sales_{i,t}}{ME_{i,t}}\right)
$$

***

### 3.2 Growth States

These variables capture firm-level growth in size and operations.

- **Book Equity Growth**

$$
beg_{i,t} = \log\left(\frac{BE_{i,t}}{BE_{i,t-1}}\right)
$$

- **Asset Growth**

$$
ag_{i,t} = \log\left(\frac{Assets_{i,t}}{Assets_{i,t-1}}\right)
$$

- **Sales Growth**

$$
sg_{i,t} = \log\left(\frac{Sales_{i,t}}{Sales_{i,t-1}}\right)
$$

***

### 3.3 Profitability States

These variables measure different notions of firm profitability.

- **Clean-Surplus Profitability**

$$
csprof_{i,t} =
\log\left(
1 +
\frac{PO_{i,t} + \Delta BE_{i,t}}{BE_{i,t-1}}
\right)
$$

This variable reflects the clean-surplus relation linking payouts and book equity growth.

- **Return on Equity**

$$
roe_{i,t} =
\log\left(
1 +
\frac{NI_{i,t}}{\tfrac{1}{2}(BE_{i,t} + BE_{i,t-1})}
\right)
$$

- **Gross Profitability**

$$
gp_{i,t} =
\log\left(
1 +
\frac{GrossProfit_{i,t}}{\tfrac{1}{2}(Assets_{i,t} + Assets_{i,t-1})}
\right)
$$

***

### 3.4 Capital Structure States

These variables describe the firm's balance sheet structure.

- **Market Leverage**

$$
lev_{i,t} =
\frac{Debt_{i,t}}{Debt_{i,t} + ME_{i,t}}
$$

- **Book Leverage**

$$
blev_{i,t} =
\frac{Debt_{i,t}}{Assets_{i,t}}
$$

- **Cash Holdings**

$$
cash_{i,t} =
\frac{Cash\ \&\ ShortTermInvestments_{i,t}}{Assets_{i,t}}
$$

Leverage ratios are kept in levels, following the specification in the paper.

***

### Remarks

- Log transformations are only applied where economically meaningful (strictly positive inputs).
- Missing values arise naturally from lag construction or accounting edge cases and are retained.
- No standardization, demeaning, or winsorization is applied at this stage.
- This step completes the data preparation required for estimating the VAR in Step 4.

After Step 3, the dataset consists of a firm-year panel with a complete, paper-consistent state vector for each firm.

In [6]:
def build_state_variables(master):
    df = master.copy()

    def safe_log(x):
        x = pd.to_numeric(x, errors="coerce")
        out = pd.Series(np.nan, index=df.index, dtype=float)
        m = x > 0
        out.loc[m] = np.log(x.loc[m])
        return out

    def safe_log1p(x):
        x = pd.to_numeric(x, errors="coerce")
        out = pd.Series(np.nan, index=df.index, dtype=float)
        m = x > -1  # log(1+x) defined iff 1+x > 0
        out.loc[m] = np.log1p(x.loc[m])
        return out

    # Ensure numeric inputs
    num_cols = [
        "BE", "ME", "PO", "sales",
        "BE_lag1", "assets", "assets_lag1", "sales_lag1",
        "dBE", "net_income", "avgBE", "gross_profit", "avgAssets",
        "debt", "CashSTInvst"
    ]
    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Valuation
    df["bm"] = safe_log(df["BE"] / df["ME"])
    df["py"] = safe_log1p(df["PO"] / df["ME"])
    df["sy"] = safe_log(df["sales"] / df["ME"])

    # Growth
    df["beg"] = safe_log(df["BE"] / df["BE_lag1"])
    df["ag"]  = safe_log(df["assets"] / df["assets_lag1"])
    df["sg"]  = safe_log(df["sales"] / df["sales_lag1"])

    # Profitability
    csprof_raw = (df["PO"] + df["dBE"]) / df["BE_lag1"]
    roe_raw    = df["net_income"] / df["avgBE"]
    gp_raw     = df["gross_profit"] / df["avgAssets"]

    # Optional paper-near floor so log1p stays well-defined
    csprof_raw = csprof_raw.clip(lower=-0.99)
    roe_raw    = roe_raw.clip(lower=-0.99)
    gp_raw     = gp_raw.clip(lower=-0.99)

    df["csprof"] = safe_log1p(csprof_raw)
    df["roe"]    = safe_log1p(roe_raw)
    df["gp"]     = safe_log1p(gp_raw)

    # Capital structure: book leverage and cash holdings
    assets = df["assets"].to_numpy(dtype=float)
    debt   = df["debt"].to_numpy(dtype=float)
    cash   = df["CashSTInvst"].to_numpy(dtype=float)
    me     = df["ME"].to_numpy(dtype=float)

    blev = np.full(len(df), np.nan, dtype=float)
    cash_ratio = np.full(len(df), np.nan, dtype=float)

    mask_assets = assets > 0
    blev[mask_assets] = debt[mask_assets] / assets[mask_assets]
    cash_ratio[mask_assets] = cash[mask_assets] / assets[mask_assets]

    df["blev"] = blev
    df["cash"] = cash_ratio

    # Capital structure: market leverage
    denom = debt + me
    lev = np.full(len(df), np.nan, dtype=float)
    mask_lev = denom > 0
    lev[mask_lev] = debt[mask_lev] / denom[mask_lev]
    df["lev"] = lev

    # Final state variable set
    state_vars = [
        "bm", "py", "sy",
        "beg", "ag", "sg",
        "csprof", "roe", "gp",
        "lev", "blev", "cash"
    ]

    df[state_vars] = df[state_vars].replace([np.inf, -np.inf], np.nan)

    return df

In [7]:
state_panel = build_state_variables(master)

In [8]:
state_vars = [
    "bm","py","sy",
    "beg","ag","sg",
    "csprof","roe","gp",
    "lev","blev","cash"
]

state_panel[state_vars].isna().mean().sort_values(ascending=False)
state_panel[state_vars].describe(percentiles=[0.01,0.05,0.5,0.95,0.99]).T

,count,mean,std,min,1%,5%,50%,95%,99%,max
bm,13489.0,-0.643032,0.831267,-8.299590,-2.859247,-2.032913,-0.596950,0.594132,1.232917,2.547247
py,14543.0,0.014274,0.102314,-3.820488,-0.329092,-0.054259,0.017790,0.088518,0.159700,0.967096
sy,13144.0,-0.241450,1.286195,-12.036174,-3.755555,-2.339821,-0.130279,1.588199,2.302337,4.281099
beg,12328.0,0.085793,0.289089,-5.970282,-0.573990,-0.202607,0.064847,0.432645,1.074398,5.797227
ag,12498.0,0.083192,0.231808,-2.238647,-0.374759,-0.126563,0.051414,0.398195,0.899081,5.707593
sg,12024.0,0.066941,0.349923,-8.357161,-0.847749,-0.236899,0.057385,0.414336,0.952304,7.695724
csprof,12421.0,0.104384,0.414975,-4.605170,-0.957675,-0.198805,0.115632,0.438862,1.015416,5.797227
roe,12387.0,0.097319,0.357710,-4.605170,-0.657288,-0.119440,0.116365,0.329200,0.597848,4.622996
gp,10905.0,0.267667,0.205973,-4.605170,0.000471,0.039634,0.232570,0.628373,0.909762,1.656442
lev,13346.0,0.316963,0.234233,0.000000,0.000091,0.007603,0.278579,0.766331,0.894739,0.987947


In [11]:
state_panel.groupby("year")[state_vars].count().tail()

,bm,py,sy,beg,ag,sg,csprof,roe,gp,lev,blev,cash
year,,,,,,,,,,,,
2021,513,535,505,492,498,486,498,496,435,515,515,497
2022,519,545,524,505,510,497,510,509,445,522,522,507
2023,504,531,511,494,498,496,498,498,434,507,507,495
2024,506,532,510,493,499,501,499,498,433,508,508,494
2025,507,536,511,494,498,494,498,497,431,508,508,494


In [14]:
# ============================================================
# Winsorize State Variables by YEAR (1% / 99%)
# ============================================================

STATE_VARS_WINSOR = [
    "bm", "py", "sy",
    "beg", "ag", "sg",
    "csprof", "roe", "gp"
]

def winsorize_by_year(df, cols, year_col="year", lower=0.01, upper=0.99):
    
    df = df.copy()
    
    for col in cols:
        
        lower_bound = df.groupby(year_col)[col].transform(
            lambda x: x.quantile(lower)
        )
        
        upper_bound = df.groupby(year_col)[col].transform(
            lambda x: x.quantile(upper)
        )
        
        df[col] = df[col].clip(lower_bound, upper_bound)
    
    return df


state_panel = winsorize_by_year(
    state_panel,
    STATE_VARS_WINSOR,
    year_col="year"
)

## Step 4: Estimation of the Firm-Level State VAR


In this step, we estimate the **dynamic law of motion** for the firm-level state variables constructed in Step 3. The objective is to characterize how valuation, growth, profitability, and capital structure jointly evolve over time and to obtain **long-horizon expectations** required for valuation and duration calculations.

Following the paper, the state dynamics are modeled using a **pooled first-order vector autoregression (VAR(1))**:

$X_{i,t+1} = \mu + \Phi X_{i,t} + \varepsilon_{i,t+1},$

where $ X_{i,t} $ denotes the vector of state variables for firm $ i $ at time $ t $, $ \mu $ is a vector of intercepts, $ \Phi $ is the transition matrix, and $ \varepsilon_{i,t+1} $ is an innovation term.

***

### State Vector Specification

The baseline VAR includes a a carefully selected set of core state variables:

- **Book-to-market ($ bm $)** — valuation anchor  
- **Payout yield ($ py $)** — payout policy  
- **Sales yield ($ sy $)** — scale-adjusted valuation  
- **Asset growth ($ ag $)** — investment dynamics  
- **Return on equity ($ roe $)** — profitability  
- **Market leverage ($ lev $)** — capital structure  
- **Book equity growth ($ beg $)** — long-run firm growth
- **Clean-surplus profitability ($ csprof $)** — payout-generating profitability

These variables are included to ensure internally consistent forecasting of future payouts in the valuation identity.

***

### Estimation Strategy

The VAR is estimated on an **unbalanced firm-year panel** using pooled ordinary least squares. To ensure reliable estimation:

- Observations must have valid state values at both $ t $ and $ t+1 $.
- Firms are required to have a minimum time-series length to enter the estimation sample.
- All state variables are demeaned and standardized using pooled moments for estimation purposes only and are transformed back to levels for forecasting and valuation.

This approach improves numerical stability and ensures comparability across state variables with different units and scales.

***

### Role in the Overall Framework

The baseline VAR includes a carefully selected set of core state variables that
balance parsimony with economic content:

- Book-to-market (bm) — valuation anchor  
- Payout yield (py) — payout policy  
- Sales yield (sy) — scale-adjusted valuation  
- Asset growth (ag) — investment dynamics  
- Return on equity (roe) — profitability  
- Market leverage (lev) — capital structure  
- Book equity growth (beg) — long-run firm growth  
- Clean-surplus profitability (csprof) — payout-generating profitability  

These variables jointly capture the main channels through which firm characteristics
affect expected cash flows and discount rates and ensure internally consistent
forecasting of future payouts in the valuation identity.


In [ ]:
# ============================================================
# STEP 5: Extended pooled VAR(1) + forecasting utilities
# ============================================================

# ----------------------------
# 5.1 State vector (extended, paper-consistent)
# ----------------------------
# Include beg and csprof because they enter the cashflow identity in Step 6.
var_states = ["bm", "py", "sy", "ag", "roe", "lev", "beg", "csprof"]

# ----------------------------
# 5.2 Build VAR sample (t -> t+1) with strict validity
# ----------------------------
df_var = state_panel.sort_values(["firm_id", "year"]).copy()

# create leads
for v in var_states:
    df_var[f"{v}_lead"] = df_var.groupby("firm_id")[v].shift(-1)

# keep only rows where ALL states exist at t and t+1
req_cols = var_states + [f"{v}_lead" for v in var_states]
df_var = df_var.dropna(subset=req_cols).copy()

# minimum time-series length per firm (strict, paper style)
min_T = 10
firm_counts = df_var.groupby("firm_id").size()
valid_firms = firm_counts[firm_counts >= min_T].index
df_var = df_var[df_var["firm_id"].isin(valid_firms)].copy()

print("STEP 5 VAR sample:")
print(f"  firms: {df_var['firm_id'].nunique()}")
print(f"  obs  : {len(df_var)}")

# ----------------------------
# 5.3 Pooled standardization (on X_t)
# ----------------------------
Z = df_var[var_states].astype(float)
Z_lead = df_var[[f"{v}_lead" for v in var_states]].astype(float)

mu = Z.mean()
sigma = Z.std().replace(0, np.nan)

Zs = (Z - mu) / sigma
Zs_lead = (Z_lead - mu.values) / sigma.values

# ----------------------------
# 5.4 Estimate pooled VAR(1): Z_{t+1} = c + Phi Z_t + u_{t+1}
# ----------------------------
Y = Zs_lead.to_numpy(dtype=float)                # (N x k)
X = sm.add_constant(Zs.to_numpy(dtype=float))    # (N x (k+1))

var_model = sm.OLS(Y, X).fit()

Phi = var_model.params[1:, :].T                 # (k x k)
const = var_model.params[0, :]                  # (k,)

phi_df = pd.DataFrame(Phi, index=var_states, columns=var_states)

# ----------------------------
# 5.5 Stability diagnostics
# ----------------------------
eigvals = np.linalg.eigvals(Phi)
max_eig = float(np.max(np.abs(eigvals)))

print("Extended VAR(1) estimated")
print(f"  Max |eigenvalue|: {max_eig:.4f}")

# ----------------------------
# 5.6 Forecast utilities for Step 6
# ----------------------------
I = np.eye(len(var_states))

# steady state of standardized VAR: zbar = (I - Phi)^{-1} const
zbar = np.linalg.solve(I - Phi, const)

def to_standardized_state(row):
    """Convert a row with raw state variables into standardized z_t."""
    x = row[var_states].to_numpy(dtype=float)
    return (x - mu.to_numpy(dtype=float)) / sigma.to_numpy(dtype=float)

def forecast_states(z0, H):
    """
    Produce E_t[z_{t+h}] for h=1..H in standardized units.
    Returns array shape (H, k).
    """
    out = np.zeros((H, len(var_states)), dtype=float)
    z = z0.copy()
    for h in range(1, H + 1):
        z = const + Phi @ z
        out[h-1, :] = z
    return out

def forecast_states_closedform(z0, H):
    """
    Closed-form version using Phi^h and steady state.
    out[h-1] = Phi^h z0 + (I - Phi^h) zbar
    """
    out = np.zeros((H, len(var_states)), dtype=float)
    for h in range(1, H + 1):
        Phi_h = np.linalg.matrix_power(Phi, h)
        out[h-1, :] = Phi_h @ z0 + (I - Phi_h) @ zbar
    return out

# Optional: a quick view of Phi
phi_df.round(3)


STEP 5 VAR sample:
  firms: 481
  obs  : 9399
Extended VAR(1) estimated
  Max |eigenvalue|: 0.9499


,bm,py,sy,ag,roe,lev,beg,csprof
bm,0.841,-0.016,-0.038,0.024,-0.009,0.055,-0.007,-0.020
py,0.023,0.134,0.014,-0.050,0.045,-0.055,0.009,0.059
sy,-0.057,-0.007,0.925,0.027,0.001,0.027,0.022,-0.017
ag,-0.144,-0.017,-0.066,0.011,0.068,-0.050,0.079,-0.060
roe,-0.180,0.020,-0.003,-0.041,0.319,0.003,0.125,-0.045
lev,-0.017,-0.010,-0.022,0.021,0.020,0.940,0.013,-0.022
beg,-0.267,-0.040,-0.018,0.089,-0.028,0.072,-0.092,-0.039
csprof,-0.267,0.012,-0.030,-0.007,0.137,0.032,-0.024,0.031


## Step 5:  Valuation Identity, Discount Rate, and Equity Duration


### Step 5: Endogenous Discount Rates and Equity Duration

This step constructs firm-year specific **equity discount rates** and **equity duration**
using a valuation-identity approach. The procedure follows a present-value framework in
which the market-to-book ratio equals the discounted value of **expected future payouts
to equityholders**, where expectations are formed using the state dynamics estimated in
Step 4.

***

#### 5.1 Valuation Identity

For each firm $ i $ and year $ t $, the market value of equity relative to book equity
satisfies the identity

$\frac{ME_{i,t}}{BE_{i,t}} = \sum_{h=1}^{\infty} \mathbb{E}_t \left[ \frac{PO_{i,t+h}}{BE_{i,t}} \right] \exp(- r_{i,t} \, h),$

where  
- $ ME_{i,t} $ denotes market equity,  
- $ BE_{i,t} $ denotes book equity,  
- $ PO_{i,t+h} $ denotes payouts to equityholders, and  
- $ r_{i,t} $ is a firm-year specific discount rate.

The discount rate $ r_{i,t} $ is **endogenously determined** such that the valuation
identity holds exactly for each firm-year observation.

***

#### 5.2 Forecasting Expected Payouts

Expected future payouts are constructed from forecasted firm-level state variables
obtained from the VAR estimated in Step 4. Two key state variables govern the payout
process:

- **Book equity growth** $ beg_{i,t} $, which determines the evolution of the scale of
  book equity over time,
- **Clean-surplus profitability** $ csprof_{i,t} $, which governs the share of book
  equity distributed to equityholders.

The payout-to-book ratio at horizon $ h $ is defined as

$\frac{PO_{i,t+h}}{BE_{i,t+h-1}} = \max\left\{ 0,\; \exp(csprof_{i,t+h}) - 1 \right\},$

ensuring that payouts represent **non-negative distributions to equityholders**, while
negative implied values are interpreted as net equity issuance rather than cash payouts.

Expected payouts relative to initial book equity are then given by

$\frac{PO_{i,t+h}}{BE_{i,t}} = \frac{PO_{i,t+h}}{BE_{i,t+h-1}} \cdot \exp\!\left( \sum_{j=1}^{h-1} beg_{i,t+j} \right).$

***

#### 5.3 Finite-Horizon Approximation and Terminal Value

The infinite sum in the valuation identity is approximated using a **finite forecast
horizon $ H $** combined with a conservative terminal value:

$\frac{ME_{i,t}}{BE_{i,t}} = \sum_{h=1}^{H} \mathbb{E}_t \left[ \frac{PO_{i,t+h}}{BE_{i,t}} \right] \exp(- r_{i,t} h) + TV_{i,t}(r_{i,t}).$

The terminal value is specified as a **level perpetuity without growth**, based on the
long-run expected payout ratio,

$TV_{i,t}(r) = \frac{\overline{po}}{\exp(r) - 1} \cdot \exp\!\left( \sum_{j=1}^{H} beg_{i,t+j} \right) \exp(- r H),$

where $ \overline{po} = \max\{0, \exp(csprof_{\infty}) - 1\} $ is the steady-state
payout-to-book ratio implied by the VAR. This conservative terminal specification avoids
explosive valuations and ensures numerically stable discount-rate solutions.

***

#### 5.4 Solving for the Discount Rate

For each firm-year observation, the discount rate $ r_{i,t} $ is obtained as the unique
solution to

$f(r) = \sum_{h=1}^{H} \mathbb{E}_t \left[ \frac{PO_{i,t+h}}{BE_{i,t}} \right] \exp(- r h) + TV_{i,t}(r) - \frac{ME_{i,t}}{BE_{i,t}} = 0.$

The root is computed using a robust bracketing method. Firm-year observations for which
no valid solution exists are excluded from the analysis.

***

#### 5.5 Equity Duration

Equity duration measures the **payout-weighted average maturity** of expected equity cash
flows and is defined as

$Dur_{i,t} = \frac{1}{ME_{i,t}/BE_{i,t}} \left[ \sum_{h=1}^{H} h \cdot \mathbb{E}_t \left[ \frac{PO_{i,t+h}}{BE_{i,t}} \right] \exp(- r_{i,t} h) + Dur^{TV}_{i,t} \right],$

where the terminal contribution equals

$Dur^{TV}_{i,t} = \left( H + \frac{1}{\exp(r_{i,t}) - 1} \right) TV_{i,t}(r_{i,t}).$

This measure captures the effective maturity of equity cash flows, analogous to Macaulay
duration for fixed-income securities.

***

#### 5.6 Sample Construction and Diagnostics

Equity duration and discount rates are computed for firm-year observations with positive
market and book equity, valid VAR forecasts, and a well-defined solution to the valuation
identity. The resulting panel exhibits substantial cross-sectional and time-series
variation. Discount rates and equity duration are negatively correlated, consistent with
asset pricing theory, and the valuation identity is satisfied up to numerical precision.


In [17]:
# ============================================================
# STEP 5: Firm-Level State VAR(1) + forecasting utilities
# (paper-near implementation, no standardization)
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm

# ----------------------------
# 5.1 Full state vector
# ----------------------------
var_states = [
    "bm", "py", "sy",
    "beg", "ag", "sg",
    "csprof", "roe", "gp",
    "lev", "blev", "cash"
]

# choose your time column here
TIME_COL = "year"   # change to "year" if needed

# ----------------------------
# 5.2 Build VAR sample (t -> t+1)
# ----------------------------
df_var = state_panel.sort_values(["firm_id", TIME_COL]).copy()

# create one-period-ahead leads
for v in var_states:
    df_var[f"{v}_lead"] = df_var.groupby("firm_id")[v].shift(-1)

# keep only rows where all states exist at t and t+1
req_cols = var_states + [f"{v}_lead" for v in var_states]
df_var = df_var.dropna(subset=req_cols).copy()

# optional: minimum time-series length per firm
min_T = 10
firm_counts = df_var.groupby("firm_id").size()
valid_firms = firm_counts[firm_counts >= min_T].index
df_var = df_var[df_var["firm_id"].isin(valid_firms)].copy()

print("STEP 5 VAR sample:")
print(f"  firms: {df_var['firm_id'].nunique()}")
print(f"  obs  : {len(df_var)}")

# ----------------------------
# 5.3 Estimate pooled VAR(1) equation by equation
#     X_{t+1} = c + Phi X_t + u_{t+1}
# ----------------------------
k = len(var_states)

X_raw = df_var[var_states].astype(float)
X_design = sm.add_constant(X_raw, has_constant="add")

Phi = np.zeros((k, k), dtype=float)
const = np.zeros(k, dtype=float)
var_results = {}

for i, v in enumerate(var_states):
    y = df_var[f"{v}_lead"].astype(float)

    res = sm.OLS(y, X_design, missing="drop").fit()
    var_results[v] = res

    const[i] = res.params["const"]
    Phi[i, :] = res.params[var_states].to_numpy(dtype=float)

phi_df = pd.DataFrame(Phi, index=var_states, columns=var_states)
const_s = pd.Series(const, index=var_states, name="const")

print("\nVAR(1) estimated equation-by-equation.")
print("\nIntercepts:")
display(const_s.round(4))

print("\nTransition matrix Phi:")
display(phi_df.round(4))

# ----------------------------
# 5.4 Stability diagnostics
# ----------------------------
eigvals = np.linalg.eigvals(Phi)
max_eig = float(np.max(np.abs(eigvals)))

print("\nStability diagnostics")
print("---------------------")
print("Eigenvalues:")
print(np.round(eigvals, 4))
print(f"Max |eigenvalue|: {max_eig:.4f}")

if max_eig >= 1:
    print("WARNING: VAR is not stable (or is very close to instability).")
else:
    print("VAR appears stable.")

# ----------------------------
# 5.5 Long-run mean / steady state
#     xbar = (I - Phi)^(-1) c
# ----------------------------
I = np.eye(k)

try:
    xbar = np.linalg.solve(I - Phi, const)
    xbar_s = pd.Series(xbar, index=var_states, name="steady_state")
    print("\nImplied steady state:")
    display(xbar_s.round(4))
except np.linalg.LinAlgError:
    xbar = None
    print("\nSteady state could not be computed because (I - Phi) is singular.")

# ----------------------------
# 5.6 Forecast utilities
# ----------------------------
def row_to_state_vector(row, states=var_states):
    """
    Convert one row of df_var / state_panel into a raw state vector x_t.
    """
    return row[states].to_numpy(dtype=float)


def forecast_states_iterative(x0, H, Phi=Phi, const=const):
    """
    Iterative forecast:
        x_{t+h} = const + Phi @ x_{t+h-1}
    Returns array of shape (H, k), for h=1..H.
    """
    out = np.zeros((H, len(x0)), dtype=float)
    x = x0.copy()

    for h in range(H):
        x = const + Phi @ x
        out[h, :] = x

    return out


def forecast_states_closedform(x0, H, Phi=Phi, const=const):
    """
    Closed-form forecast:
        x_{t+h} = Phi^h x0 + (I - Phi^h) xbar
    Requires a stable VAR and computable steady state xbar.
    Returns array of shape (H, k), for h=1..H.
    """
    if xbar is None:
        raise ValueError("xbar is not available because (I - Phi) could not be inverted.")

    k = len(x0)
    I = np.eye(k)
    out = np.zeros((H, k), dtype=float)

    for h in range(1, H + 1):
        Phi_h = np.linalg.matrix_power(Phi, h)
        out[h - 1, :] = Phi_h @ x0 + (I - Phi_h) @ xbar

    return out


def forecast_dataframe_from_row(row, H, method="iterative"):
    """
    Convenience wrapper:
    returns a DataFrame with forecasted states for horizons 1..H.
    """
    x0 = row_to_state_vector(row)

    if method == "iterative":
        arr = forecast_states_iterative(x0, H)
    elif method == "closedform":
        arr = forecast_states_closedform(x0, H)
    else:
        raise ValueError("method must be 'iterative' or 'closedform'")

    out = pd.DataFrame(arr, columns=var_states)
    out.insert(0, "horizon", np.arange(1, H + 1))
    return out


# ----------------------------
# 5.7 Optional quick summary of equation fit
# ----------------------------
fit_summary = pd.DataFrame({
    "R2": {v: var_results[v].rsquared for v in var_states},
    "Adj_R2": {v: var_results[v].rsquared_adj for v in var_states},
    "N": {v: int(var_results[v].nobs) for v in var_states},
})
print("\nEquation fit summary:")
display(fit_summary.round(4))

STEP 5 VAR sample:
  firms: 412
  obs  : 7884

VAR(1) estimated equation-by-equation.

Intercepts:


bm       -0.0468
py        0.0231
sy       -0.0041
beg      -0.0028
ag        0.0488
sg        0.0420
csprof    0.0236
roe       0.0246
gp        0.0275
lev       0.0386
blev      0.0286
cash      0.0196
Name: const, dtype: float64


Transition matrix Phi:


,bm,py,sy,beg,ag,sg,csprof,roe,gp,lev,blev,cash
bm,0.8487,-0.1043,-0.0069,0.1609,0.0593,0.0276,-0.1619,0.0680,-0.1382,0.0311,0.0063,-0.1951
py,0.0053,0.2909,0.0017,0.0062,-0.0195,-0.0019,0.0245,0.0349,-0.0105,-0.0187,0.0056,-0.0035
sy,-0.1008,-0.2858,0.9600,0.1019,0.2844,-0.0783,-0.0276,-0.0621,-0.1540,0.0007,-0.0279,-0.2633
beg,-0.0729,-0.1097,-0.0057,0.1207,0.0465,-0.0044,-0.1430,0.0566,0.0294,0.1070,-0.0543,0.0027
ag,-0.0188,-0.2511,-0.0106,0.0319,0.0286,0.0360,-0.0251,0.0753,0.0391,-0.0891,0.0495,0.0225
sg,-0.0279,-0.3495,-0.0254,0.0358,0.2735,-0.0818,-0.0191,-0.0487,-0.0067,0.0316,-0.0413,-0.0527
csprof,-0.0810,-0.0902,-0.0060,-0.1204,-0.0053,-0.0047,0.1480,0.2386,0.0106,0.0538,-0.0385,-0.0101
roe,-0.0618,-0.1250,0.0029,-0.0745,-0.0255,0.0127,0.1062,0.2999,0.0348,0.0212,-0.0130,-0.0072
gp,-0.0106,-0.0367,0.0027,0.0056,-0.0546,-0.0073,0.0029,-0.0371,0.8994,-0.0198,0.0043,0.0069
lev,-0.0054,-0.0313,0.0017,-0.0010,0.0295,0.0272,-0.0122,0.0322,-0.0424,0.8822,0.0307,-0.0534



Stability diagnostics
---------------------
Eigenvalues:
[-0.136 +0.j     -0.0067+0.j      0.06  +0.j      0.192 +0.j
  0.3515+0.0363j  0.3515-0.0363j  0.9644+0.j      0.7883+0.j
  0.9147+0.j      0.8402+0.j      0.8971+0.j      0.8692+0.j    ]
Max |eigenvalue|: 0.9644
VAR appears stable.

Implied steady state:


bm       -0.5136
py        0.0248
sy       -0.3899
beg       0.0613
ag        0.0610
sg        0.0568
csprof    0.1035
roe       0.0959
gp        0.1953
lev       0.3389
blev      0.2904
cash      0.1084
Name: steady_state, dtype: float64


Equation fit summary:


,R2,Adj_R2,N
bm,0.7560,0.7557,7884
py,0.1451,0.1438,7884
sy,0.8656,0.8654,7884
beg,0.0732,0.0718,7884
ag,0.0901,0.0887,7884
sg,0.1100,0.1086,7884
csprof,0.1575,0.1562,7884
roe,0.2467,0.2456,7884
gp,0.8838,0.8837,7884
lev,0.8164,0.8161,7884


#### VAR Estimation Results

The estimated firm-level VAR appears well behaved. The transition matrix shows economically plausible persistence patterns across the state variables, with particularly strong persistence in valuation and balance-sheet variables such as book-to-market, sales yield, leverage, and cash holdings. Growth and profitability variables are less persistent, as expected.

Importantly, the VAR is stable: the largest eigenvalue in absolute value is 0.9644, which is below one. This implies that multi-period forecasts are well defined and that the implied steady state exists. Overall, the estimated system provides a reasonable basis for generating the expected future state paths required in the duration calculation.

## OTHER

In [ ]:
# Coverage diagnostics by year vs EURO500 net-payout universe
# Denominator: unique firm_id in euro500_netpayout per year
# Numerator: unique firm_id with solved duration (status == "ok") per year
import matplotlib.pyplot as plt

# universe by year from Step 1/2 input
univ = euro500_netpayout[["firm_id", "year"]].dropna().copy()
univ["firm_id"] = univ["firm_id"].astype(str)
univ["year"] = pd.to_numeric(univ["year"], errors="coerce").astype("Int64")
univ = univ.dropna(subset=["year"]).copy()
univ["year"] = univ["year"].astype(int)
univ = univ[univ["year"] >= 1999].copy()

universe_by_year = (
    univ.drop_duplicates(["firm_id", "year"])
    .groupby("year")["firm_id"]
    .nunique()
    .rename("universe_n")
)

# solved durations by year
ok = duration_panel_v3.query("status=='ok'")[["firm_id", "year"]].dropna().copy()
ok["firm_id"] = ok["firm_id"].astype(str)
ok["year"] = pd.to_numeric(ok["year"], errors="coerce").astype("Int64")
ok = ok.dropna(subset=["year"]).copy()
ok["year"] = ok["year"].astype(int)
ok = ok[ok["year"] >= 1999].copy()

ok_by_year = (
    ok.drop_duplicates(["firm_id", "year"])
    .groupby("year")["firm_id"]
    .nunique()
    .rename("ok_n")
)

coverage_by_year = (
    pd.concat([universe_by_year, ok_by_year], axis=1)
    .fillna(0)
    .astype({"universe_n": int, "ok_n": int})
    .reset_index()
    .sort_values("year")
)
coverage_by_year["coverage_rate"] = coverage_by_year["ok_n"] / coverage_by_year["universe_n"]


print("Coverage vs euro500_netpayout universe (head):")
print(coverage_by_year.head())

fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.plot(coverage_by_year["year"], coverage_by_year["coverage_rate"], marker="o", linewidth=1.6, color="tab:blue")
ax1.set_ylim(0, 1.02)
ax1.set_xlabel("Year")
ax1.set_ylabel("Coverage Rate", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(coverage_by_year["year"], coverage_by_year["universe_n"], linestyle="--", linewidth=1.2, color="tab:gray", label="Universe N")
ax2.plot(coverage_by_year["year"], coverage_by_year["ok_n"], linestyle="-", linewidth=1.2, color="tab:green", label="Duration N")
ax2.set_ylabel("Number of Equities")

lines, labels = ax2.get_legend_handles_labels()
ax2.legend(lines, labels, loc="lower right", frameon=False)

plt.title("Step 5 Coverage by Year: Duration vs EURO500 Net-Payout Universe")
plt.tight_layout()
plt.show()

coverage_by_year


## Step 6: Build Euro500 x NetPayout Duration Panel (Quarterly Mapping)


In [ ]:
# Step 6: Map annual NetPayout durations to all Euro500 quarters of the same year

# 6.1 Build clean annual duration panel (firm_id x year)
dur_annual = duration_panel_v3.query("status == 'ok'").copy()
dur_annual = dur_annual.rename(columns={
    "equity_duration": "Duration_NP",
    "discount_rate": "discount_rate_NP",
})

dur_annual["firm_id"] = dur_annual["firm_id"].astype(str).str.strip()
dur_annual["year"] = pd.to_numeric(dur_annual["year"], errors="coerce").astype("Int64")

dur_annual = (
    dur_annual.dropna(subset=["firm_id", "year", "Duration_NP"])
    .groupby(["firm_id", "year"], as_index=False)
    .agg(
        Duration_NP=("Duration_NP", "median"),
        discount_rate_NP=("discount_rate_NP", "median"),
        pv_check=("pv_check", "median"),
    )
)
dur_annual["year"] = dur_annual["year"].astype(int)
dur_annual["YEAR"] = dur_annual["year"].astype("Int64")

# 6.2 Load quarterly Euro500 base panel (target shape for ECB shocks workflow)
base_q = load_parquet("euro500").copy()
if "firm_id" not in base_q.columns:
    raise KeyError("euro500 must contain firm_id")

# derive year strictly from effective_date
if "year" not in base_q.columns:
    if "effective_date" not in base_q.columns:
        raise KeyError("euro500 needs effective_date to derive year")
    base_q["year"] = pd.to_datetime(base_q["effective_date"], errors="coerce").dt.year

base_q["firm_id"] = base_q["firm_id"].astype(str).str.strip()
base_q["year"] = pd.to_numeric(base_q["year"], errors="coerce").astype("Int64")
base_q = base_q.dropna(subset=["firm_id", "year"]).copy()
base_q["year"] = base_q["year"].astype(int)
base_q["YEAR"] = base_q["year"].astype("Int64")

# Keep representative quarterly fields used downstream
keep_cols = [c for c in [
    "firm_id", "RIC", "RIC_current", "quarter", "date", "asof_date", "formation_date", "effective_date",
    "name", "hq_country", "hq_code", "trbc_sector", "trbc_sector_code", "mcap_eur", "year", "YEAR"
] if c in base_q.columns]

base_q = base_q[keep_cols].drop_duplicates([c for c in ["firm_id", "quarter"] if c in keep_cols]).copy()

# 6.3 Map annual duration to each quarter (join key: firm_id x year)
dur_q = base_q.merge(
    dur_annual,
    on=["firm_id", "year", "YEAR"],
    how="left",
    validate="m:1",
)

# Optional compatibility alias for ECBShocks notebook (expects this column name)
dur_q["Duration_DCF_Macaulay"] = dur_q["Duration_NP"]
dur_q["status"] = np.where(dur_q["Duration_NP"].notna(), "ok", "missing")

# 6.4 Diagnostics
firm_year_cov = dur_q[["firm_id", "year", "Duration_NP"]].drop_duplicates(["firm_id", "year"]).copy()
coverage_year = (
    firm_year_cov.groupby("year", as_index=False)
    .agg(
        universe_n=("firm_id", "nunique"),
        with_duration_n=("Duration_NP", lambda s: s.notna().sum()),
    )
)
coverage_year["coverage_rate"] = coverage_year["with_duration_n"] / coverage_year["universe_n"]

print("Annual duration sample (firm_id-year):", dur_annual.shape)
print("Quarterly mapped panel (Euro500 x quarter):", dur_q.shape)
print("Coverage by year (first rows):")
print(coverage_year.head())

# 6.5 Save outputs
# (A) firm-year durations
save_parquet(dur_annual, "equity_duration_panel")
# (B) euro500-quarter mapped panel for ECB shocks ingestion
save_parquet(dur_q, "EQDuration_NetPayout")
